# 08. 외식업 10종 오프라인 분석 도구

**구현 범위:** 검증된 CSV·JSON을 읽는 5개 Python 조회 함수, JSON 호출 계약, 지원 범위 검증, 출처와 호출 로그다. 외식업 10종의 지표 정의·매출 집계·모형 결과·업종 전체 기울기 차이 검정·품질 요약을 조회한다. 실제 LLM·자연어 계획기·파서·웹 서비스·MCP 서버는 구현하지 않았다.

순서: 입력의 버전 확인 → 지표 정의와 JSON Schema → 고정 함수 → JSON 호출 실행기 → 사람이 선택한 호출 예시 → 성공·거부 검증. final 01→02→03→04→05→06→07을 먼저 실행한다. 06의 결과 묶음이 있으면 해시·수렴을 확인해 보조 GEE 조회를 제공하며, 묶음 전체가 없으면 FE 조회만 제공한다. 데이터·모형을 수정하거나 다시 적합하지 않으며 네트워크 요청·자유 코드 실행·임의 SQL/SPARQL 실행은 없다. 생성 파일은 `results/main/baseline/07_*`에 저장한다.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, math, platform, uuid
import numpy as np
import pandas as pd
from jsonschema import Draft202012Validator, ValidationError
from IPython.display import display, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'AGENTS.md').is_file() and (p / 'data/raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('AGENTS.md와 data/raw가 있는 프로젝트 루트에서 실행하세요.')
REPORTS = ROOT / 'results/main/baseline'
REPORTS.mkdir(parents=True, exist_ok=True)
RUN_ID = str(uuid.uuid4())
INPUTS = {
    'panel': ROOT / 'data/analysis/final/food_industry_panel.csv',
    'catalog': ROOT / 'data/analysis/final/food_industry_catalog.csv',
    'scope_summary': REPORTS / '08_food_scope_metadata.json',
    'scope_checks': REPORTS / '08_validation_checks.csv',
    'model_results': REPORTS / '09_industry_effects.csv',
    'model_samples': REPORTS / '09_model_samples.csv',
    'model_omnibus': REPORTS / '09_omnibus_tests.csv',
    'model_summary': REPORTS / '09_analysis_summary.json',
    'model_checks': REPORTS / '09_fwl_validation.csv',
    'graph_summary': REPORTS / '06_knowledge_graph_summary.json',
    'graph_manifest': REPORTS / '06_source_manifest.csv',
    'graph_checks': REPORTS / '06_validation.csv',
    'ontology': ROOT / 'data/analysis/final/knowledge_graph/ontology.ttl',
}
GEE_INPUTS = {name: REPORTS / filename for name, filename in {
    'gee_results': '10_gee_industry_effects.csv',
    'gee_summary': '10_gee_analysis_summary.json',
    'gee_diagnostics': '10_gee_diagnostics.csv',
    'gee_checks': '10_gee_implementation_checks.csv',
    'gee_sample_flow': '10_gee_sample_flow.csv',
}.items()}
gee_presence = [path.exists() for path in GEE_INPUTS.values()]
assert not any(gee_presence) or all(gee_presence), '06 GEE 결과 묶음이 일부만 있습니다. 06을 완료하거나 묶음을 제외하세요.'
GEE_AVAILABLE = all(gee_presence)
if GEE_AVAILABLE:
    INPUTS.update(GEE_INPUTS)
def file_sha256(path):
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()
INPUT_HASHES = {name: file_sha256(path) for name, path in INPUTS.items()}
frames = {name: pd.read_csv(path, dtype={'area_id': str, 'quarter': str, 'industry_id': str})
          for name, path in INPUTS.items() if path.suffix == '.csv'}
metadata = {name: json.loads(path.read_text(encoding='utf-8'))
            for name, path in INPUTS.items() if path.suffix == '.json'}
panel, catalog = frames['panel'], frames['catalog'].sort_values('industry_id')
models, samples, omnibus = frames['model_results'], frames['model_samples'], frames['model_omnibus']
scope_summary, model_summary, graph_summary = [metadata[k] for k in ['scope_summary', 'model_summary', 'graph_summary']]
print('로컬 입력:', len(INPUTS), '개 | 보조 GEE:', GEE_AVAILABLE, '| Python:', platform.python_version())

로컬 입력: 18 개 | 보조 GEE: True | Python: 3.12.14


## 1. 저장된 검증 결과의 입력 버전 확인

04의 외식업 10종 코드·명칭과 07의 직접 입력 해시를 확인한다. 05가 사용한 패널 및 결과 CSV 해시가 현재 파일과 같아야 조회를 허용한다. 선택적 06 GEE 묶음도 같은 패널·업종 목록과 결과 해시를 확인한다. 일부 파일만 바뀌거나 다른 실행 결과가 섞이면 중단한다. 해시는 파일 버전 일치 확인용이며 자료의 완전성·인과 식별을 보증하지 않는다.

수집 완료 상태는 지정 상세 수집 대상의 완료로 해석한다. 품질 응답에는 서울 목록 전체와 부분 그래프의 공식·잠정 연결 수를 별도 분모로 반환하며, 현재 시설 좌표 확보와 과거 분기별 시설 위치 검증을 구분한다.

In [2]:
DISTRICTS = ['종로구', '마포구']
INDUSTRY_NAMES = dict(catalog[['industry_id', 'industry']].to_numpy())
INDUSTRY_IDS, INDUSTRIES = list(INDUSTRY_NAMES), list(INDUSTRY_NAMES.values())
QUARTERS = [f'{year}{q}' for year in range(2023, 2026) for q in range(1, 5)]
OUTCOMES = ['log_sales', 'night_share_pct']
FE_SCENARIOS = ['main_500', 'radius_300', 'radius_800', 'balanced_500', 'stores_500', 'total_stores_500']
SCENARIO_LABELS = {'main_500': '500m 주분석', 'radius_300': '300m 거리 민감도',
    'radius_800': '800m 거리 민감도', 'balanced_500': '500m 균형표본 민감도',
    'stores_500': '500m 일반점포 수 통제 민감도',
    'total_stores_500': '500m 전체점포 수 통제 민감도',
    'gee_within_between_500': '500m within-between GEE 보조 분석'}
SCENARIOS = FE_SCENARIOS + (['gee_within_between_500'] if GEE_AVAILABLE else [])
METRICS = ['total_sales', 'night_share_pct', 'known_performance_count', 'model_association']
MODEL_KEY = ['scenario', 'outcome', 'industry_id']
assert INDUSTRY_IDS == [f'CS100{i:03d}' for i in range(1, 11)]
assert catalog['included'].eq(True).all()
assert set(map(tuple, panel[['industry_id', 'industry']].drop_duplicates().values)) == set(INDUSTRY_NAMES.items())
assert scope_summary['industries'] == catalog[['industry_id', 'industry']].to_dict('records')
assert set(panel['district']) == set(DISTRICTS) and set(panel['quarter']) == set(QUARTERS)
assert not panel.duplicated(['area_id', 'industry_id', 'quarter']).any()
assert len(panel) == panel['area_id'].nunique() * 10 * 12 == scope_summary['panel_rows']
assert panel['sales_record_present'].eq(panel['total_sales'].notna()).all()
assert {'regular_store_count', 'franchise_store_count', 'total_store_count'} <= set(panel.columns)
assert int(panel['total_sales'].notna().sum()) == scope_summary['observed_sales_rows']
assert scope_summary['panel_sha256'] == INPUT_HASHES['panel']
assert scope_summary['catalog_sha256'] == INPUT_HASHES['catalog']
assert model_summary['input_sha256'] == INPUT_HASHES['panel']
for name in ['model_results', 'model_samples', 'model_omnibus', 'model_checks']:
    assert model_summary['output_sha256'][INPUTS[name].relative_to(ROOT).as_posix()] == INPUT_HASHES[name]
for row in frames['graph_manifest'].itertuples(index=False):
    assert file_sha256(ROOT / row.path) == row.sha256, f'07 RDF 입력 버전이 달라졌습니다: {row.path}'
for name in ['panel', 'catalog']:
    record = frames['graph_manifest'].loc[lambda t: t['source'].eq(name)]
    assert len(record) == 1 and record.iloc[0]['sha256'] == INPUT_HASHES[name]
assert set(graph_summary['scope']['industry_ids']) == set(INDUSTRY_IDS)
assert graph_summary['kopis_collection_run_id'] == scope_summary['exposure_snapshot_linkage']['collection_run_id']
assert graph_summary['kopis_collection_status'] == scope_summary['exposure_snapshot_linkage']['collection_status']
assert graph_summary['collected_performance_detail_rows'] == scope_summary['exposure_snapshot_linkage']['performance_detail_rows']
assert graph_summary['collected_facility_detail_rows'] == scope_summary['exposure_snapshot_linkage']['facility_detail_rows']
assert graph_summary['seoul_performance_ids_in_input'] == scope_summary['exposure_snapshot_linkage']['performance_ids']
assert graph_summary['input_official_links'] == scope_summary['exposure_snapshot_linkage']['official_link_ids']
assert graph_summary['input_provisional_links'] == scope_summary['exposure_snapshot_linkage']['provisional_name_link_ids']
assert graph_summary['official_links'] + graph_summary['provisional_links'] == graph_summary['performance_ids']
assert graph_summary['seoul_refresh_run_id'] == scope_summary['seoul_refresh_run_id']
assert graph_summary['sales_observations'] == len(panel)
assert graph_summary['sales_values'] == int(panel['total_sales'].notna().sum())
assert frames['graph_checks']['status'].eq('passed').all() and graph_summary['all_validations_passed']
assert frames['scope_checks']['passed'].eq(True).all()
assert len(frames['scope_checks']) == scope_summary['validation_checks_passed']
assert frames['model_checks']['passed'].eq(True).all()
assert not models.duplicated(MODEL_KEY).any()
assert not samples.duplicated(['scenario', 'outcome']).any()
assert not omnibus.duplicated(['scenario', 'outcome']).any()
assert models['industry_id'].isin(INDUSTRY_IDS).all()
assert models['scenario'].isin(FE_SCENARIOS).all() and models['outcome'].isin(OUTCOMES).all()
display(catalog[['industry_id', 'industry']])
display(pd.DataFrame([{'validation': name, 'passed': len(frames[key]), 'total': len(frames[key])}
    for name, key in [('04 범위·패널 검사', 'scope_checks'), ('05 고정 기준선 FWL 대조', 'model_checks'),
                      ('07 RDF·SPARQL 검증', 'graph_checks')]]))

,industry_id,industry
0,CS100001,한식음식점
1,CS100002,중식음식점
2,CS100003,일식음식점
3,CS100004,양식음식점
4,CS100005,제과점
5,CS100006,패스트푸드점
6,CS100007,치킨전문점
7,CS100008,분식전문점
8,CS100009,호프-간이주점
9,CS100010,커피-음료


,validation,passed,total
0,04 범위·패널 검사,18,18
1,05 고정 기준선 FWL 대조,12,12
2,07 RDF·SPARQL 검증,14,14


### 선택적 GEE 결과 묶음 검증

보조 GEE 결과를 제공할 때는 패널·업종 목록과 05 FE 결과의 해시, 06 GEE 결과 CSV의 해시 및 수렴·수용 검사를 함께 확인한다. 이전 실행 파일이 남아 있으면 새 패널의 결과로 사용하지 않는다.

In [3]:
if GEE_AVAILABLE:
    gee_summary = metadata['gee_summary']
    gee_input_hashes = {key.replace('\\', '/'): value for key, value in gee_summary['input_sha256'].items()}
    for name in ['panel', 'catalog']:
        assert gee_input_hashes[INPUTS[name].relative_to(ROOT).as_posix()] == INPUT_HASHES[name]
    assert gee_summary['fe_input_sha256'] == INPUT_HASHES['model_results']
    for name in ['gee_results', 'gee_diagnostics', 'gee_checks', 'gee_sample_flow']:
        assert gee_summary['output_sha256'][INPUTS[name].name] == INPUT_HASHES[name]
    assert frames['gee_diagnostics']['accepted'].eq(True).all()
    assert frames['gee_diagnostics']['converged'].eq(True).all()
    assert frames['gee_checks']['passed'].eq(True).all()
    assert not frames['gee_results'].duplicated(MODEL_KEY).any()
    assert frames['gee_results']['industry_id'].isin(INDUSTRY_IDS).all()
    display(frames['gee_diagnostics'][['outcome', 'nobs', 'area_clusters', 'accepted', 'converged']])
else:
    print('06 GEE 결과 묶음이 없어 FE 조회만 제공합니다.')

,outcome,nobs,area_clusters,accepted,converged
0,log_sales,7646,134,True,True
1,night_share_pct,7646,134,True,True


**해석:** 아래의 모형 조회는 현재 패널과 같은 버전으로 계산된 저장 결과를 돌려준다. 회귀 재적합이나 새 질문에 대한 모형 선택은 수행하지 않는다. 매출 조회는 선택 상권의 제공 행을 합한다. 상권 간 중첩을 제거한 합계가 아니므로 점포 기준 중복 없는 자치구 총매출로 해석하지 않는다.

## 2. 공통 응답 형식과 지표 정의

모든 응답은 JSON으로 직렬화할 수 있어야 한다. 자료에 없는 숫자는 `null`로 표시하고 `NaN`·무한대는 내보내지 않는다. 매출 조회는 공식 업종명, 모형 조회는 같은 업종의 공식 코드(`CS100001`~`CS100010`)를 사용한다. 코드·명칭 대응은 04의 목록과 대조한다.

In [4]:
def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if value is pd.NA or value is pd.NaT:
        return None
    return value

def source_ref(name, **details):
    return {'path': INPUTS[name].relative_to(ROOT).as_posix(), 'sha256': INPUT_HASHES[name], **details}

SCOPE = {'districts': DISTRICTS, 'industries': INDUSTRY_NAMES, 'quarters': QUARTERS,
         'time_unit': 'quarter', 'causal_identification': False}
LIMITATIONS = ['분기 단위이므로 일별 효과를 식별하지 않는다.',
    '미제공 매출은 0원이 아니다. 제공 행 합계는 자치구 전체 매출이 아니다.',
    f"서울 전체 입력의 시설명 잠정 연결은 {graph_summary['input_provisional_links']:,}건, 현재 부분 그래프의 잠정 연결은 {graph_summary['provisional_links']:,}건이다. 각 범위의 분모를 구분한다.",
    f"상세 수집 {graph_summary['collected_performance_detail_rows']:,}건 / 서울 목록 {graph_summary['seoul_performance_ids_in_input']:,}건이다. 수집 완료는 지정 대상의 완료이며 서울 전체의 공식 ID 연결 완료를 뜻하지 않는다.",
    f"현재 부분 그래프의 좌표 미확인 시설은 {graph_summary['facilities_without_coordinates']}개다. 과거 분기별 시설 위치는 검증하지 않았다.",
    '현재 상권 경계와 시설 위치를 과거 분기에 공통 적용한다.',
    '등록 공연 ID 수는 상연 회차·관객 수가 아니다. 상권별 합계는 중복될 수 있다.',
    '공간 상관·자료 선택·측정오차와 인과 식별은 해결되지 않았다.',
    '두 업종 분석 후 확대한 외식업 10종 비교는 탐색 분석이다.',
    '일반점포가 0이어도 프랜차이즈가 있으면 전체점포는 양수일 수 있다. 이를 매출 오류로 분류하지 않는다.',
    '추정매출의 점포 모집단은 확인되지 않았으므로 점포당 매출을 계산하거나 해석하지 않는다.']
SUPPORTED_CONTEXT = {'time_unit': 'quarter', 'claim_type': 'association',
                     'performance_measure': 'registered_performance_ids', 'missing_sales': 'preserve_missing'}

def response(data, source_names, caveats):
    result = json_safe({'status': 'ok', 'data': data, 'scope': SCOPE,
                       'sources': [source_ref(name) for name in source_names], 'caveats': caveats})
    json.dumps(result, ensure_ascii=False, allow_nan=False)
    return result

In [5]:
DEFINITIONS = {
    'total_sales': {
        'label': '분기 총매출', 'unit': 'KRW', 'time_unit': 'quarter',
        'source_column': '당월_매출_금액',
        'calculation': '선택한 상권·업종·분기의 제공된 total_sales 합계; 전체 미제공이면 null',
        'source_origin': '서울시 상권분석서비스 추정매출 분기 자료; 04 전처리 및 02 공간 연결',
        'caution': '원본 열에 당월이라는 명칭이 있어도 관측 단위는 분기다. 미제공 금액은 0원으로 채우지 않는다.'},
    'night_share_pct': {
        'label': '21~24시 매출 비중', 'unit': 'percent', 'time_unit': 'quarter',
        'source_column': '시간대_21~24_매출_금액 / 당월_매출_금액',
        'calculation': '총매출과 야간매출이 함께 제공된 행에서 100 × 야간매출 합계 / 총매출 합계',
        'source_origin': '서울시 상권분석서비스 추정매출; 04 노트북',
        'caution': '행별 야간비중의 단순 평균이 아니다. 같은 행으로 분자와 분모를 구성하고 분모가 0이면 null이다.'},
    'known_performance_count': {
        'label': '반경 내 기간 중복 등록 공연 ID 수', 'unit': 'unique_performance_ids', 'time_unit': 'quarter',
        'source_column': 'known_performance_count_300m/500m/800m',
        'calculation': '현재 상권 폴리곤까지 직선거리 이내인 유효 좌표 시설의 분기 중복 공연 ID 고유 개수',
        'source_origin': 'KOPIS 공연 목록·상세·시설 좌표 및 현재 서울 상권 경계; 02·04 노트북',
        'caution': '상연 횟수·관객 수가 아니다. 공식 ID 연결과 시설명 임시 연결을 구분한다. 현재 좌표 누락 수는 품질 조회로 확인하며 과거 시설 위치는 미검증이다. 상권별 수를 더하면 같은 공연이 중복될 수 있다.'},
    'model_association': {
        'label': '저장 모형의 업종별 공연 활동 기울기', 'unit': 'outcome_dependent', 'time_unit': 'quarter',
        'source_column': '09_industry_effects.csv 및 선택적 10_gee_industry_effects.csv',
        'calculation': 'log(1+등록 공연 수) 1단위 변화와 선택 업종 결과변수의 관련성; 업종 전체 차이는 별도 omnibus 검정 조회',
        'source_origin': '05 외식업 10종 FE 및 선택적 06 within-between GEE 보조 분석',
        'caution': 'log_sales는 log(1+원), night_share_pct는 퍼센트포인트 단위다. 1공연 증가 효과나 인과효과가 아니다.'},
}
metric_table = pd.DataFrame([{'metric': key, 'label': value['label'], 'unit': value['unit'], 'time_unit': value['time_unit']}
                             for key, value in DEFINITIONS.items()])
display(metric_table)

,metric,label,unit,time_unit
0,total_sales,분기 총매출,KRW,quarter
1,night_share_pct,21~24시 매출 비중,percent,quarter
2,known_performance_count,반경 내 기간 중복 등록 공연 ID 수,unique_performance_ids,quarter
3,model_association,저장 모형의 업종별 공연 활동 기울기,outcome_dependent,quarter


**해석:** 야간 비중의 분자는 21~24시 매출이다. 저녁 전체·심야 전체를 임의로 확장하지 않는다. 지표별 단위와 분모를 응답에 남겨, 이후 AI가 연결되더라도 서로 다른 수치를 같은 지표로 설명하지 않도록 한다.

### RDF 정의와 실제 연결

07의 작은 `ontology.ttl`을 RDF로 파싱해 한국어 `rdfs:label`·`rdfs:comment`와 속성의 domain/range를 조회한다. 총매출은 `ex:totalSales`에 직접 대응한다. 공연 수는 독립된 RDF 지표 속성이 없으므로 공연-시설-거리 관계와 분기 중복 규칙을 근거로 반환한다. 야간 비중·점포수·05·06 회귀 결과는 현재 RDF에 포함되어 있지 않으며, RDF에 있는 것처럼 응답하지 않는다. 전체 관측 그래프를 적재하거나 임의 SPARQL을 받지 않는다.

In [6]:
from rdflib import Graph, Namespace, RDFS
EX = Namespace('https://example.org/grad-study/')
ontology = Graph().parse(INPUTS['ontology'], format='turtle')
RDF_TERMS = {
    'total_sales': [EX.totalSales, EX.salesProvided, EX.salesUnit, EX.KRW, EX.quarterlySales],
    'known_performance_count': [EX.activeQuarter, EX.nearFacility, EX.linkedPerformance,
                                EX.distanceMetres, EX.quarterOverlap, EX.polygon500m, EX.metre],
}

def read_rdf_definition(metric):
    terms = RDF_TERMS.get(metric, [])
    definitions = []
    for term in terms:
        definitions.append({'uri': str(term), 'labels': [str(v) for v in ontology.objects(term, RDFS.label)],
            'comments': [str(v) for v in ontology.objects(term, RDFS.comment)],
            'domains': [str(v) for v in ontology.objects(term, RDFS.domain)],
            'ranges': [str(v) for v in ontology.objects(term, RDFS.range)]})
    return {'available': bool(terms), 'definitions': definitions,
            'mapping': ('직접 속성 ex:totalSales' if metric == 'total_sales' else
                        '관계와 규칙에서 도출하는 DISTINCT 공연 ID 집계; 직접 지표 속성 없음' if terms else
                        '현재 RDF 범위 밖; CSV 기반 정의·계산 사용')}

assert list(ontology.objects(EX.totalSales, RDFS.label))
assert list(ontology.objects(EX.KRW, RDFS.label)), '07 단위 정의가 ontology.ttl에 필요합니다.'
display(pd.DataFrame(read_rdf_definition('total_sales')['definitions'])[['uri', 'labels', 'ranges']])

,uri,labels,ranges
0,https://example.org/grad-study/totalSales,[분기 총매출액],[http://www.w3.org/2001/XMLSchema#decimal]
1,https://example.org/grad-study/salesProvided,[매출 원천 행 제공 여부],[http://www.w3.org/2001/XMLSchema#boolean]
2,https://example.org/grad-study/salesUnit,[매출액 단위],[https://example.org/grad-study/Unit]
3,https://example.org/grad-study/KRW,[원],[]
4,https://example.org/grad-study/quarterlySales,[상권·업종·분기 총매출],[]


**해석:** RDF는 지표와 규칙의 의미를 제공하고, 금액과 모형 결과는 CSV에서 읽는다. 이 연결은 실제 RDF 문서를 읽는 기능이며, 자연어 질의 생성이나 RDF에서 회귀를 수행하는 기능은 아니다.

## 3. JSON 도구 계약과 지원 범위

5개 함수는 미리 정한 enum과 고정된 파일만 사용한다. 범위 밖 enum은 `unsupported`, 누락 필드·잘못된 타입·알 수 없는 인수는 `invalid_request`로 반환한다. 호출자가 일별·인과효과·실제 상연 횟수·미제공 0원 처리를 구조화된 `request_context`로 지정하면 `unsupported`가 된다. **자연어에서 이런 의도를 자동 추출하는 기능은 없다.**

In [7]:
def enum_property(values):
    return {'type': 'string', 'enum': values}

def object_schema(properties):
    return {'type': 'object', 'properties': properties, 'required': list(properties), 'additionalProperties': False}

PARAMETER_SCHEMAS = {
    'get_metric_definition': object_schema({'metric': enum_property(METRICS)}),
    'summarize_sales': object_schema({'district': enum_property(DISTRICTS), 'industry': enum_property(INDUSTRIES),
                                      'quarter': enum_property(QUARTERS)}),
    'get_model_result': object_schema({'outcome': enum_property(OUTCOMES), 'scenario': enum_property(SCENARIOS),
                                       'industry_id': enum_property(INDUSTRY_IDS)}),
    'get_industry_omnibus': object_schema({'outcome': enum_property(OUTCOMES),
                                          'scenario': enum_property(FE_SCENARIOS)}),
    'get_quality_summary': object_schema({}),
}
TOOL_DESCRIPTIONS = {
    'get_metric_definition': '지표의 단위, 분기 기준, 산식, 출처, 지원 범위를 조회한다.',
    'summarize_sales': '선택한 자치구·업종·분기의 제공 매출 합계와 합계의 비율 방식 야간 비중을 조회한다.',
    'get_model_result': '저장된 특정 시나리오·결과변수·업종 코드의 기울기, 신뢰구간, p값과 표본 수를 조회한다.',
    'get_industry_omnibus': '05의 업종별 기울기가 모두 같다는 귀무가설의 공동 F 검정과 Holm p값을 조회한다.',
    'get_quality_summary': '통과한 구조·모형 검증과 별도로 미해결 자료·해석 한계를 조회한다.',
}
CONTEXT_SCHEMA = object_schema({
    'time_unit': enum_property(['quarter', 'daily']),
    'claim_type': enum_property(['association', 'causal_effect']),
    'performance_measure': enum_property(['registered_performance_ids', 'show_runs']),
    'missing_sales': enum_property(['preserve_missing', 'fill_zero']),
})
ENVELOPE_SCHEMA = object_schema({'tool': enum_property(list(PARAMETER_SCHEMAS)),
    'arguments': {'type': 'object'}, 'request_context': CONTEXT_SCHEMA})
# request_context는 생략할 수 있으며 생략 시 적용한 기본 전제를 응답에 명시한다.
ENVELOPE_SCHEMA['required'] = ['tool', 'arguments']
for schema in [*PARAMETER_SCHEMAS.values(), CONTEXT_SCHEMA, ENVELOPE_SCHEMA]:
    Draft202012Validator.check_schema(schema)
TOOL_CONTRACTS = {'schema_version': '0.3-final-all10food', 'status': 'offline_callable_tools_only',
    'llm_connected': False, 'web_service_built': False, 'default_context': SUPPORTED_CONTEXT,
    'industry_catalog': INDUSTRY_NAMES, 'optional_gee_available': GEE_AVAILABLE,
    'request_envelope_schema': ENVELOPE_SCHEMA,
    'tools': [{'type': 'function', 'function': {'name': name, 'description': TOOL_DESCRIPTIONS[name], 'parameters': schema}}
              for name, schema in PARAMETER_SCHEMAS.items()]}
(REPORTS / '07_tool_contracts.json').write_text(json.dumps(TOOL_CONTRACTS, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
display(pd.DataFrame([{'tool': name, 'required_arguments': ', '.join(schema['required']) or '(없음)'}
                     for name, schema in PARAMETER_SCHEMAS.items()]))

,tool,required_arguments
0,get_metric_definition,metric
1,summarize_sales,"district, industry, quarter"
2,get_model_result,"outcome, scenario, industry_id"
3,get_industry_omnibus,"outcome, scenario"
4,get_quality_summary,(없음)


## 4. 고정 조회 함수

원본 매출이 없는 행은 그대로 남기고, 제공·미제공·실제 0원 행 수를 각각 반환한다. 야간 비중은 두 금액이 함께 관측된 같은 행에서 계산하며 해당 분모와 행 수도 함께 내보낸다. 자료가 전혀 제공되지 않거나 분모가 0인 경우 숫자를 만들어 넣지 않는다.

In [8]:
def summarize_night_ratio(frame):
    paired = frame['total_sales'].notna() & frame['night_sales'].notna()
    denominator = frame.loc[paired, 'total_sales'].sum(min_count=1)
    numerator = frame.loc[paired, 'night_sales'].sum(min_count=1)
    if pd.isna(denominator):
        status, reason, value = 'unsupported', 'no_common_provided_denominator', None
    elif denominator <= 0:
        status, reason, value = 'unsupported', 'nonpositive_denominator', None
    else:
        status, reason, value = 'ok', None, 100 * numerator / denominator
    return json_safe({'status': status, 'reason': reason, 'value_pct': value,
                      'paired_rows': paired.sum(), 'numerator_krw': numerator, 'denominator_krw': denominator})

In [9]:
def validate_parameters(tool, arguments):
    Draft202012Validator(PARAMETER_SCHEMAS[tool]).validate(arguments)

def get_metric_definition(metric):
    validate_parameters('get_metric_definition', {'metric': metric})
    sources = ['model_results', 'model_summary'] if metric == 'model_association' else ['panel', 'scope_summary', 'catalog']
    if metric == 'model_association' and GEE_AVAILABLE:
        sources += ['gee_results', 'gee_summary']
    rdf_definition = read_rdf_definition(metric)
    if rdf_definition['available']:
        sources.append('ontology')
    return response({'metric': metric, **DEFINITIONS[metric], 'rdf_definition': rdf_definition}, sources, [DEFINITIONS[metric]['caution']])

def summarize_sales(district, industry, quarter):
    args = {'district': district, 'industry': industry, 'quarter': quarter}
    validate_parameters('summarize_sales', args)
    subset = panel.loc[panel['district'].eq(district) & panel['industry'].eq(industry) & panel['quarter'].eq(quarter)]
    provided = subset['total_sales'].notna()
    night = summarize_night_ratio(subset)
    data = {'selection': args, 'unit': 'KRW', 'time_unit': 'quarter', 'panel_rows': len(subset),
        'areas': subset['area_id'].nunique(), 'provided_sales_rows': provided.sum(),
        'missing_sales_rows': (~provided).sum(), 'zero_sales_rows': subset['total_sales'].eq(0).sum(),
        'total_sales_krw': subset['total_sales'].sum(min_count=1),
        'night_share_pct': night['value_pct'], 'night_share_unit': 'percent', 'night_share_method': 'ratio_of_sums_on_same_rows',
        'night_share_validation': {'status': night['status'], 'reason': night['reason']},
        'night_share_paired_rows': night['paired_rows'], 'night_share_numerator_krw': night['numerator_krw'],
        'night_share_denominator_krw': night['denominator_krw'],
        'negative_total_sales_rows': subset['total_sales'].lt(0).sum(),
        'night_exceeds_total_rows': (subset['night_sales'] > subset['total_sales']).sum()}
    caveats = [DEFINITIONS['total_sales']['caution'], DEFINITIONS['night_share_pct']['caution'],
               '금액은 선택 상권의 제공 행 합계이며 상권 간 중첩을 제거하지 않았다. 점포 기준 중복 없는 자치구 총매출이나 미제공 행을 포함한 전체 매출로 해석하지 않는다.']
    return response(data, ['panel', 'scope_summary', 'catalog'], caveats)

**해석:** `provided_sales_rows + missing_sales_rows = panel_rows`이며, 0원 행 수는 실제 0원 관측만 센다. 야간 비중은 `night_share_numerator_krw / night_share_denominator_krw × 100`으로 재확인할 수 있다. 평균비중과 구분하기 위해 계산 방식을 응답에 포함한다.

In [10]:
def get_model_result(outcome, scenario, industry_id):
    args = {'outcome': outcome, 'scenario': scenario, 'industry_id': industry_id}
    validate_parameters('get_model_result', args)
    is_gee = scenario == 'gee_within_between_500'
    source = 'gee_results' if is_gee else 'model_results'
    table = frames[source]
    selected = table.loc[table['outcome'].eq(outcome) & table['scenario'].eq(scenario)
                         & table['industry_id'].eq(industry_id)]
    if len(selected) != 1 or (is_gee and not bool(selected.iloc[0]['estimated'])):
        return {'status': 'unsupported', 'reason': 'requested_saved_result_unavailable', 'selection': args}
    row = selected.iloc[0]
    sample = (frames['gee_diagnostics'].loc[lambda t: t['outcome'].eq(outcome)] if is_gee else
              samples.loc[samples['outcome'].eq(outcome) & samples['scenario'].eq(scenario)])
    correction_column = 'p_fdr_gee_exploratory' if is_gee else 'p_fdr_primary'
    unit = 'log(1+매출 원) 변화 / log(1+등록 공연 수) 1단위' if outcome == 'log_sales' else '퍼센트포인트 / log(1+등록 공연 수) 1단위'
    data = {'selection': args, 'scenario_label': SCENARIO_LABELS[scenario], 'industry': INDUSTRY_NAMES[industry_id], 'estimator': 'GEE' if is_gee else 'FE',
        'store_control_column': model_summary['store_control_columns'].get(scenario),
        'radius_m': 500 if is_gee else row['radius_m'], 'component': 'within' if is_gee else 'fixed_effect_slope',
        'coefficient': row['coefficient'], 'standard_error': row['standard_error'],
        'confidence_interval': {'level': 0.95, 'low': row['ci_low'], 'high': row['ci_high']},
        'p_value': row['p_value'], 'p_fdr': row[correction_column], 'p_fdr_source_column': correction_column,
        'nobs_industry': row['nobs_industry'], 'area_clusters_industry': row['area_clusters_industry'],
        'nobs_total': row['nobs_total'], 'area_clusters_total': row['area_clusters_total'], 'unit': unit,
        'estimator_note': ('상권×업종 및 업종×분기 고정효과; 상권 군집 t 추론' if not is_gee else
            '상권×업종 내 변화의 기울기; within-between Gaussian identity GEE, 상권 exchangeable 군집·robust 정규근사'),
        'sample_metadata': sample.iloc[0].to_dict() if len(sample) == 1 else None,
        'saved_result': source_ref(source, csv_line=int(selected.index[0]) + 2), 'causal_identification': False}
    notes = ['저장된 결과 조회이며 재적합·자동 모형 선택을 하지 않는다.',
        '계수는 관련성이다. 1공연 증가 효과나 인과효과로 해석하지 않는다.',
        '업종별 표본 크기와 공동 추정 전체 표본·군집 수를 구분한다.',
        ('GEE BH p값은 2개 결과변수×10개 within 기울기의 보조 탐색 검정군이다.' if is_gee else
         'FE BH p값은 main_500의 2개 결과변수×10개 기울기 검정군이다. 민감도 모형의 보정 p값 null은 0이나 유의함을 뜻하지 않는다.')]
    sources = [source, 'catalog', 'gee_summary', 'gee_diagnostics'] if is_gee else [source, 'catalog', 'model_samples', 'model_summary']
    return response(data, sources, notes)

In [11]:
def get_industry_omnibus(outcome, scenario):
    args = {'outcome': outcome, 'scenario': scenario}
    validate_parameters('get_industry_omnibus', args)
    selected = omnibus.loc[omnibus['outcome'].eq(outcome) & omnibus['scenario'].eq(scenario)]
    if len(selected) != 1 or selected.iloc[0]['status'] != 'estimated':
        return {'status': 'unsupported', 'reason': 'requested_saved_test_unavailable', 'selection': args}
    row = selected.iloc[0]
    data = {'selection': args, 'null_hypothesis': '선택 모형의 추정 업종 기울기가 모두 같다',
        'estimated_industries': row['estimated_industries'], 'f_statistic': row['f_statistic'],
        'df_num': row['df_num'], 'df_denom': row['df_denom'], 'p_value': row['p_value'],
        'p_holm_primary': row['p_holm_primary'], 'causal_identification': False,
        'saved_result': source_ref('model_omnibus', csv_line=int(selected.index[0]) + 2)}
    return response(data, ['model_omnibus', 'model_summary'], [
        '귀무가설은 모든 기울기가 0이라는 뜻이 아니다. 어느 두 업종이 다른지 지정하지 않는다.',
        'Holm 보정은 main_500의 두 결과변수에 대한 공동 검정군에만 적용한다.',
        '저장한 FE 공동 검정이며 인과효과나 새 사후 쌍별 검정을 제공하지 않는다.'])

In [12]:
def get_quality_summary():
    validate_parameters('get_quality_summary', {})
    store_definition_examples = panel['total_sales'].gt(0) & panel['regular_store_count'].eq(0) & panel['total_store_count'].gt(0)
    positive_sales_zero_total = panel['total_sales'].gt(0) & panel['total_store_count'].eq(0)
    data = {'store_count_semantics': {
        'regular_store_count': '일반점포 수: STOR_CO',
        'franchise_store_count': '프랜차이즈점포 수: FRC_STOR_CO',
        'total_store_count': '전체점포 수: SIMILR_INDUTY_STOR_CO',
        'deprecated_alias': 'store_count는 일반점포 수의 이전 이름이며 전체점포 수가 아니다.',
        'positive_sales_zero_regular_positive_total_rows': int(store_definition_examples.sum()),
        'positive_sales_zero_total_rows': int(positive_sales_zero_total.sum()),
        'interpretation': '일반점포 0·전체점포 양수·양의 매출은 프랜차이즈가 있는 정의 차이 사례이며 오류가 아니다.',
        'sales_per_store_available': False,
        'sales_population_caution': '추정매출 점포 모집단 미확인: 점포당 매출 지표를 제공하지 않는다.'},
        'validation': {
        'scope_checks_passed': int(frames['scope_checks']['passed'].sum()),
        'scope_checks_total': len(frames['scope_checks']),
        'model_reference_checks_passed': int(frames['model_checks']['passed'].sum()),
        'graph_checks_passed': int(frames['graph_checks']['status'].eq('passed').sum()),
        'accepted_gee_models': int(frames['gee_diagnostics']['accepted'].sum()) if GEE_AVAILABLE else None,
        'input_panel_matches_graph_and_models': True},
        'coverage': {'panel_rows': len(panel), 'industry_count': len(INDUSTRY_IDS),
            'observed_sales_rows': int(panel['total_sales'].notna().sum()),
            'missing_sales_rows': int(panel['total_sales'].isna().sum()),
            'industry_catalog': catalog[['industry_id', 'industry', 'sales_value_coverage_pct']].to_dict('records'),
            'full_linkage_snapshot': scope_summary['exposure_snapshot_linkage'],
            'linkage_coverage_by_scope': graph_summary['coverage_by_scope'],
            'partial_graph': {key: graph_summary[key] for key in ['performance_ids', 'official_links',
                'provisional_links', 'facilities_without_coordinates', 'performances_without_coordinates',
                'excluded_performance_ids', 'input_performance_ids_without_facility']}},
        'unresolved_limitations': LIMITATIONS,
        'implementation_status': {'llm_connected': False, 'web_service_built': False,
            'natural_language_parser_built': False, 'offline_tools_only': True, 'optional_gee_available': GEE_AVAILABLE}}
    sources = ['panel', 'scope_summary', 'scope_checks', 'catalog', 'model_checks', 'model_summary',
               'graph_summary', 'graph_manifest', 'graph_checks']
    if GEE_AVAILABLE:
        sources += ['gee_summary', 'gee_diagnostics', 'gee_checks']
    return response(data, sources, ['서울 전체 연결 입력과 연구 범위 부분 그래프의 분모를 구분한다.',
        '구조·계산 검증의 통과와 자료의 완전성·인과 식별은 별개다.'])

**해석:** 각 업종의 관련성과 업종 사이 차이는 서로 다른 질문이다. 업종 기울기의 유의성만 비교해 업종 간 차이라고 말하지 않는다. 공동 검정은 추정한 업종의 기울기가 모두 같다는 귀무가설을 검정하며, 어느 두 업종이 다른지 지정하지 않는다. GEE는 평균구조·추론이 다른 보조 결과로 구분한다.

## 5. JSON 호출 실행기와 감사 로그

실행기는 JSON Schema로 검사한 뒤 5개 함수의 고정 등록표에서만 호출한다. JSON에 적힌 함수명·경로·코드를 임의 실행하지 않는다. 각 호출에 UTC 시각, 실행 ID, 구조화 인수·문맥, 모든 입력 파일 해시와 응답 요약을 남긴다. 이 로그는 연구 재현용 로컬 JSONL이며 운영 서비스의 인증·권한·감사 저장소를 대신하지 않는다.

In [13]:
TOOL_FUNCTIONS = {'get_metric_definition': get_metric_definition, 'summarize_sales': summarize_sales,
                  'get_model_result': get_model_result, 'get_industry_omnibus': get_industry_omnibus,
                  'get_quality_summary': get_quality_summary}
LOG_PATH = REPORTS / '07_tool_call_log.jsonl'
LOG_PATH.write_text('', encoding='utf-8')
CALL_LOG = []
CONTEXT_REASONS = {'time_unit': 'daily_data_not_available', 'claim_type': 'causal_effect_not_identified',
                   'performance_measure': 'show_run_counts_not_available', 'missing_sales': 'missing_sales_zero_imputation_not_supported'}

def reject_nonfinite(token):
    raise ValueError(f'Non-finite JSON number is not allowed: {token}')

def execute_json_call(payload_json):
    timestamp = datetime.now(timezone.utc).isoformat()
    request, applied_context = {}, None
    try:
        request = json.loads(payload_json, parse_constant=reject_nonfinite)
        Draft202012Validator(ENVELOPE_SCHEMA).validate(request)
        tool, arguments = request['tool'], request['arguments']
        validate_parameters(tool, arguments)
        applied_context = request.get('request_context', SUPPORTED_CONTEXT).copy()
        unsupported = [CONTEXT_REASONS[k] for k, value in applied_context.items() if value != SUPPORTED_CONTEXT[k]]
        if unsupported:
            result = {'status': 'unsupported', 'reason': 'request_outside_supported_semantics',
                      'unsupported_reasons': unsupported, 'supported_context': SUPPORTED_CONTEXT}
        else:
            result = TOOL_FUNCTIONS[tool](**arguments)
    except ValidationError as error:
        result = {'status': 'unsupported' if error.validator == 'enum' else 'invalid_request',
                  'reason': 'schema_validation_failed', 'field': '.'.join(map(str, error.absolute_path)),
                  'validation_rule': error.validator, 'message': error.message}
    except (ValueError, TypeError) as error:
        result = {'status': 'invalid_request', 'reason': 'invalid_json', 'message': str(error)}
    result = json_safe({**result, 'applied_context': applied_context,
                       'run_id': RUN_ID, 'sources_version': INPUT_HASHES})
    json.dumps(result, ensure_ascii=False, allow_nan=False)
    data = result.get('data', {})
    result_summary = {'status': result['status'], 'reason': result.get('reason'),
                      'data_fields': list(data), 'panel_rows': data.get('panel_rows'), 'nobs_total': data.get('nobs_total'),
                      'night_share_status': data.get('night_share_validation', {}).get('status')}
    request_dict = request if isinstance(request, dict) else {}
    entry = json_safe({'run_id': RUN_ID, 'timestamp_utc': timestamp, 'tool': request_dict.get('tool'),
        'arguments': request_dict.get('arguments'), 'request_context': request_dict.get('request_context'),
        'applied_context': applied_context, 'input_hashes': INPUT_HASHES, 'result_summary': result_summary})
    with LOG_PATH.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(entry, ensure_ascii=False, allow_nan=False) + '\n')
    CALL_LOG.append(entry)
    return result

## 6. 사람이 선택한 구조화 호출 예시

자연어 질문 예시: **“2025년 4분기 종로구 중식음식점의 제공된 매출 합계와 21~24시 매출 비중은?”**

아래 호출은 이 질문을 보고 **사람이 미리 선택한 구조화 호출**이다. 자연어를 자동 해석해 인수를 만든 결과가 아니며, 실제 AI의 응답 품질을 평가한 사례도 아니다.

In [14]:
human_selected_call = {'tool': 'summarize_sales',
    'arguments': {'district': '종로구', 'industry': '중식음식점', 'quarter': '20254'},
    'request_context': SUPPORTED_CONTEXT.copy()}
example_result = execute_json_call(json.dumps(human_selected_call, ensure_ascii=False, allow_nan=False))
print(json.dumps(human_selected_call, ensure_ascii=False, indent=2))
display(pd.DataFrame([example_result['data']]).drop(columns=['selection']))
print('출처:', example_result['sources'][0]['path'])
display(pd.DataFrame([CALL_LOG[-1]])[['timestamp_utc', 'tool', 'arguments', 'result_summary']])
(REPORTS / '07_example_result.json').write_text(json.dumps(example_result, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

{
  "tool": "summarize_sales",
  "arguments": {
    "district": "종로구",
    "industry": "중식음식점",
    "quarter": "20254"
  },
  "request_context": {
    "time_unit": "quarter",
    "claim_type": "association",
    "performance_measure": "registered_performance_ids",
    "missing_sales": "preserve_missing"
  }
}


,unit,time_unit,panel_rows,areas,provided_sales_rows,missing_sales_rows,zero_sales_rows,total_sales_krw,night_share_pct,night_share_unit,night_share_method,night_share_validation,night_share_paired_rows,night_share_numerator_krw,night_share_denominator_krw,negative_total_sales_rows,night_exceeds_total_rows
0,KRW,quarter,74,74,13,61,0,2.000687e+10,11.821386,percent,ratio_of_sums_on_same_rows,"{'status': 'ok', 'reason': None}",13,2.365089e+09,2.000687e+10,0,0


출처: data/analysis/final/food_industry_panel.csv


,timestamp_utc,tool,arguments,result_summary
0,2026-10-06T05:53:34.268341+00:00,summarize_sales,"{'district': '종로구', 'industry': '중식음식점', 'quar...","{'status': 'ok', 'reason': None, 'data_fields'..."


3984

**해석:** 표의 매출은 제공된 상권 행의 합계이고 야간 비중은 같은 관측 행을 사용한 합계의 비율이다. 제공되지 않은 행 수도 함께 확인해야 한다. 원자료에 없는 매출을 0원으로 대체하거나 분기 금액을 일별 금액으로 나누지 않았다.

## 7. 성공·지원 거부 계약 검증

5개 함수, 외식업 10종의 매출·FE 조회, 선택적 GEE 조회를 확인한다. 업종 밖 소매업 코드와 일별·인과·회차·결측 0원 요구를 구조화 인수로 전달해 거부를 검사한다. 이는 자연어 방어력이나 운영 보안 검증이 아니다. 반환 금액·비중·기울기·공동 검정값은 원자료 및 저장 결과와 대조하고 JSON의 NaN·무한대를 금지한다.

In [15]:
def call(tool, arguments, context=None):
    request = {'tool': tool, 'arguments': arguments}
    if context is not None:
        request['request_context'] = context
    return execute_json_call(json.dumps(request, ensure_ascii=False, allow_nan=False))
checks = []
def record_check(name, passed, detail):
    checks.append({'check': name, 'passed': bool(passed), 'detail': detail})
metric_result = call('get_metric_definition', {'metric': 'total_sales'})
model_args = {'outcome': 'night_share_pct', 'scenario': 'main_500', 'industry_id': 'CS100002'}
model_result = call('get_model_result', model_args)
omnibus_result = call('get_industry_omnibus', {'outcome': 'night_share_pct', 'scenario': 'main_500'})
quality_result = call('get_quality_summary', {})
store_quality = quality_result['data']['store_count_semantics']
display(pd.DataFrame([store_quality])[['positive_sales_zero_regular_positive_total_rows',
    'positive_sales_zero_total_rows', 'sales_per_store_available']])
record_check('five_fixed_tools_return_ok', all(r['status'] == 'ok' for r in
    [example_result, metric_result, model_result, omnibus_result, quality_result]), '5개 고정 도구 정상 응답')
record_check('collection_and_graph_denominators_preserved',
    quality_result['data']['coverage']['full_linkage_snapshot']['performance_ids'] == graph_summary['seoul_performance_ids_in_input']
    and quality_result['data']['coverage']['partial_graph']['performance_ids'] == graph_summary['performance_ids']
    and quality_result['data']['coverage']['linkage_coverage_by_scope'] == graph_summary['coverage_by_scope'],
    '서울 목록·상세 수집·부분 그래프의 분모와 공식/잠정 연결 구분')
selection = panel.loc[panel['district'].eq('종로구') & panel['industry'].eq('중식음식점') & panel['quarter'].eq('20254')]
returned = example_result['data']
record_check('sales_counts_preserve_missing', returned['provided_sales_rows'] + returned['missing_sales_rows'] == len(selection)
    and returned['missing_sales_rows'] == selection['total_sales'].isna().sum(), '제공/미제공 분리; 0원 대체 없음')
common = selection['total_sales'].notna() & selection['night_sales'].notna()
expected_numerator = selection.loc[common, 'night_sales'].sum(min_count=1)
expected_denominator = selection.loc[common, 'total_sales'].sum(min_count=1)
record_check('ratio_of_sums_matches_source', np.isclose(returned['night_share_pct'], 100 * expected_numerator / expected_denominator)
    and returned['night_share_numerator_krw'] == expected_numerator
    and returned['night_share_denominator_krw'] == expected_denominator
    and returned['night_share_paired_rows'] == int(common.sum())
    and returned['total_sales_krw'] == selection['total_sales'].sum(min_count=1), '매출 합계와 야간 공통관측 분자·분모·행 수 일치')
source_model = models.loc[models['scenario'].eq('main_500') & models['outcome'].eq('night_share_pct')
                         & models['industry_id'].eq('CS100002')].iloc[0]
record_check('saved_model_values_unchanged', model_result['data']['coefficient'] == source_model['coefficient']
    and model_result['data']['confidence_interval']['low'] == source_model['ci_low']
    and model_result['data']['nobs_industry'] == source_model['nobs_industry'], '재적합 없이 저장한 계수·구간·업종 표본 반환')
source_omnibus = omnibus.loc[omnibus['scenario'].eq('main_500') & omnibus['outcome'].eq('night_share_pct')].iloc[0]
record_check('saved_omnibus_values_unchanged', omnibus_result['data']['f_statistic'] == source_omnibus['f_statistic']
    and omnibus_result['data']['p_holm_primary'] == source_omnibus['p_holm_primary'], '05 공동 F 검정 및 Holm p값 일치')
display(pd.DataFrame([model_result['data']])[['selection', 'coefficient', 'confidence_interval', 'p_value', 'p_fdr', 'nobs_industry']])

,positive_sales_zero_regular_positive_total_rows,positive_sales_zero_total_rows,sales_per_store_available
0,95,0,False


,selection,coefficient,confidence_interval,p_value,p_fdr,nobs_industry
0,"{'outcome': 'night_share_pct', 'scenario': 'ma...",-0.147106,"{'level': 0.95, 'low': -0.9158060555444252, 'h...",0.705647,0.90555,458


### 외식업 10종 및 선택적 GEE 조회 검사

업종 확장을 한식·커피의 예시만으로 판단하지 않고 10개 코드의 모형 조회와 공식 명칭의 매출 조회를 모두 확인한다. 기본 모형 밖의 보정 p값 누락은 `null`로 유지한다.

In [16]:
industry_checks = []
for industry_id, industry in INDUSTRY_NAMES.items():
    sales_response = call('summarize_sales', {**human_selected_call['arguments'], 'industry': industry})
    fe_response = call('get_model_result', {**model_args, 'industry_id': industry_id})
    industry_checks.append({'industry_id': industry_id, 'industry': industry,
        'sales_status': sales_response['status'], 'fe_status': fe_response['status']})
record_check('all_ten_sales_and_fe_tools', all(row['sales_status'] == row['fe_status'] == 'ok'
    for row in industry_checks), '10종 공식 코드·명칭 모두 조회 가능')
for scenario in ['stores_500', 'total_stores_500']:
    store_response = call('get_model_result', {**model_args, 'scenario': scenario})
    record_check(f'{scenario}_definition_label', store_response['status'] == 'ok'
        and store_response['data']['scenario_label'] == SCENARIO_LABELS[scenario]
        and store_response['data']['store_control_column'] == model_summary['store_control_columns'][scenario], '일반점포·전체점포 통제 모형 구분')
sensitivity = call('get_model_result', {**model_args, 'scenario': 'radius_300'})
record_check('unadjusted_sensitivity_p_is_null', sensitivity['data']['p_fdr'] is None, '민감도 미보정 p값은 null')
if GEE_AVAILABLE:
    gee_responses = [call('get_model_result', {**model_args, 'scenario': 'gee_within_between_500', 'industry_id': code})
                     for code in INDUSTRY_IDS]
    record_check('all_ten_gee_tools', all(r['status'] == 'ok' and r['data']['component'] == 'within'
        and r['data']['sample_metadata'] is not None for r in gee_responses), '10종 GEE within 기울기와 표본 메타데이터')
    saved = frames['gee_results'].loc[lambda t: t['outcome'].eq(model_args['outcome'])
        & t['industry_id'].eq(INDUSTRY_IDS[0])].iloc[0]
    record_check('saved_gee_values_unchanged', gee_responses[0]['data']['coefficient'] == saved['coefficient']
        and gee_responses[0]['data']['p_fdr'] == saved['p_fdr_gee_exploratory'], '저장 GEE 계수·보정 p값 일치')
display(pd.DataFrame(industry_checks))

,industry_id,industry,sales_status,fe_status
0,CS100001,한식음식점,ok,ok
1,CS100002,중식음식점,ok,ok
2,CS100003,일식음식점,ok,ok
3,CS100004,양식음식점,ok,ok
4,CS100005,제과점,ok,ok
5,CS100006,패스트푸드점,ok,ok
6,CS100007,치킨전문점,ok,ok
7,CS100008,분식전문점,ok,ok
8,CS100009,호프-간이주점,ok,ok
9,CS100010,커피-음료,ok,ok


In [17]:
# 계산 경계 사례만 메모리에서 검증한다. 실제 데이터나 도구의 입력 파일을 교체하지 않는다.
paired_fixture = pd.DataFrame({'total_sales': [100.0, 900.0, np.nan], 'night_sales': [20.0, np.nan, 999.0]})
paired_result = summarize_night_ratio(paired_fixture)
missing_result = summarize_night_ratio(pd.DataFrame({'total_sales': [np.nan], 'night_sales': [np.nan]}))
zero_result = summarize_night_ratio(pd.DataFrame({'total_sales': [0.0], 'night_sales': [0.0]}))
record_check('ratio_uses_common_observed_rows', paired_result['value_pct'] == 20.0
             and paired_result['paired_rows'] == 1 and paired_result['denominator_krw'] == 100.0,
             '분자 또는 분모만 있는 행을 함께 제외')
record_check('zero_denominator_refuses_ratio', zero_result['status'] == 'unsupported'
             and zero_result['value_pct'] is None and zero_result['denominator_krw'] == 0.0,
             '실제 총매출 0원은 보존하면서 정의되지 않는 비중 산출 거부')
record_check('all_missing_stays_null', missing_result['value_pct'] is None
             and missing_result['numerator_krw'] is None and missing_result['denominator_krw'] is None,
             '전체 미제공이면 금액·비중 모두 null; 0원 대체 없음')
record_check('rdf_definition_is_read_from_ontology', metric_result['data']['rdf_definition']['definitions'][0]['labels']
             == [str(v) for v in ontology.objects(EX.totalSales, RDFS.label)]
             and not read_rdf_definition('night_share_pct')['available'], '실제 RDF 정의 조회 및 RDF 범위 밖 지표 구분')
display(pd.DataFrame([{'case': '공통 관측 행', **paired_result}, {'case': '분모 0', **zero_result}]))

,case,status,reason,value_pct,paired_rows,numerator_krw,denominator_krw
0,공통 관측 행,ok,NaN,20.0,1,20.0,100.0
1,분모 0,unsupported,nonpositive_denominator,NaN,1,0.0,0.0


In [18]:
refusals = []
for field, value in [('time_unit', 'daily'), ('claim_type', 'causal_effect'),
                     ('performance_measure', 'show_runs'), ('missing_sales', 'fill_zero')]:
    context = {**SUPPORTED_CONTEXT, field: value}
    result = call('summarize_sales', human_selected_call['arguments'], context)
    refusals.append({'case': f'{field}={value}', 'status': result['status'], 'reason': result['unsupported_reasons'][0]})
    record_check(f'reject_{field}', result['status'] == 'unsupported', result['unsupported_reasons'][0])
outside_scope = call('summarize_sales', {**human_selected_call['arguments'], 'district': '강남구'})
outside_industry = call('get_model_result', {**model_args, 'industry_id': 'CS200001'})
unknown_argument = call('get_quality_summary', {'file_path': 'arbitrary.csv'})
record_check('strict_scope_enum', outside_scope['status'] == 'unsupported', '지원하지 않는 자치구 거부')
record_check('outside_food_industry_rejected', outside_industry['status'] == 'unsupported', '외식업 10종 밖 소매업 코드 거부')
record_check('additional_arguments_rejected', unknown_argument['status'] == 'invalid_request', '임의 파일 경로 인수 거부')
refusals.extend([{'case': 'district=강남구', 'status': outside_scope['status'], 'reason': outside_scope['reason']},
                 {'case': 'industry_id=CS200001', 'status': outside_industry['status'], 'reason': outside_industry['reason']},
                 {'case': 'unexpected file_path', 'status': unknown_argument['status'], 'reason': unknown_argument['reason']}])
display(pd.DataFrame(refusals))

,case,status,reason
0,time_unit=daily,unsupported,daily_data_not_available
1,claim_type=causal_effect,unsupported,causal_effect_not_identified
2,performance_measure=show_runs,unsupported,show_run_counts_not_available
3,missing_sales=fill_zero,unsupported,missing_sales_zero_imputation_not_supported
4,district=강남구,unsupported,schema_validation_failed
5,industry_id=CS200001,unsupported,schema_validation_failed
6,unexpected file_path,invalid_request,schema_validation_failed


**해석:** “일별”이라는 문자열을 자연어 문장에서 찾아내는 규칙을 만든 것이 아니다. 호출자가 `time_unit=daily` 같은 구조화 필드로 요청했을 때 지원 범위를 지키는지만 확인했다. 앞으로 LLM을 연결한다면 이 계약으로 인수를 생성하는 단계와 답변이 수치·출처를 보존하는 단계를 별도로 평가해야 한다.

## 8. 실행 결과·계약·로그 저장

노트북을 다시 실행하면 새 실행 ID로 이번 실행의 로그를 재생성한다. 파일 해시로 입력이 실행 전후 동일한지 확인하고, 모든 호출과 거부 기록을 엄격한 JSON으로 다시 읽는다. 이 결과는 오프라인 도구 계층의 준비 상태이며 LLM·웹 서비스 완성 여부와 구분한다.

In [19]:
disk_logs = [json.loads(line, parse_constant=reject_nonfinite) for line in LOG_PATH.read_text(encoding='utf-8').splitlines()]
record_check('every_call_logged', len(disk_logs) == len(CALL_LOG) and len(CALL_LOG) > 30, '모든 정상·거부 호출마다 한 행 기록')
record_check('all_log_entries_have_provenance', all(entry['input_hashes'] == INPUT_HASHES
             and entry['timestamp_utc'] and entry['run_id'] == RUN_ID for entry in disk_logs), 'UTC 시각·인수·입력 해시·결과 요약')
record_check('inputs_unchanged', all(file_sha256(path) == INPUT_HASHES[name] for name, path in INPUTS.items()), f'입력 {len(INPUTS)}개 읽기 전용 확인')
for value in [TOOL_CONTRACTS, example_result, model_result, quality_result, omnibus_result, *disk_logs]:
    json.loads(json.dumps(value, ensure_ascii=False, allow_nan=False), parse_constant=reject_nonfinite)
record_check('strict_json_round_trip', True, 'NaN·Infinity 없는 JSON 저장 및 재읽기')
check_table = pd.DataFrame(checks)
check_table.to_csv(REPORTS / '07_contract_checks.csv', index=False, encoding='utf-8-sig')
summary = {'status': 'offline_callable_tools_only', 'llm_connected': False, 'web_service_built': False,
    'natural_language_parser_built': False, 'network_requests': 0,
    'rdf_definition_source': INPUTS['ontology'].relative_to(ROOT).as_posix(),
    'rdf_scope': '총매출 속성 및 공연기간·거리관계·단위/집계규칙 정의; 야간비중/점포/회귀결과는 RDF에 없음', 'tool_count': len(TOOL_FUNCTIONS),
    'scope': SCOPE, 'optional_gee_available': GEE_AVAILABLE,
    'model_scenario_labels': {key: SCENARIO_LABELS[key] for key in SCENARIOS},
    'store_count_semantics': quality_result['data']['store_count_semantics'],
    'model_default_source': INPUTS['model_results'].relative_to(ROOT).as_posix(),
    'seoul_refresh_run_id': scope_summary.get('seoul_refresh_run_id'),
    'kopis_collection_run_id': scope_summary['exposure_snapshot_linkage'].get('collection_run_id'),
    'kopis_collection_status': graph_summary['kopis_collection_status'],
    'collected_performance_detail_rows': graph_summary['collected_performance_detail_rows'],
    'collected_facility_detail_rows': graph_summary['collected_facility_detail_rows'],
    'linkage_coverage_by_scope': graph_summary['coverage_by_scope'],
    'run_id': RUN_ID, 'calls': len(CALL_LOG),
    'call_status_counts': pd.Series([entry['result_summary']['status'] for entry in CALL_LOG]).value_counts().to_dict(),
    'contract_checks_passed': int(check_table['passed'].sum()), 'contract_checks_total': len(check_table),
    'input_sources': {name: source_ref(name) for name in INPUTS},
    'limitations': LIMITATIONS,
    'next_work': ['LLM 연결 시 구조화 인수 생성과 수치·출처 보존 평가', '필요한 경우 인증·권한을 갖춘 실제 서비스 계층 별도 구현']}
(REPORTS / '07_tool_readiness_summary.json').write_text(json.dumps(json_safe(summary), ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
display(check_table)
assert check_table['passed'].all(), '도구 계약 검증 실패 항목을 확인하세요.'
display(Markdown(f"**완료 범위:** 고정 도구 {len(TOOL_FUNCTIONS)}개, 호출 {len(CALL_LOG)}건, 계약 검사 {len(check_table)}개 통과. "
                '실제 LLM 연결·자연어 파서·웹 서비스는 구현하지 않았다.'))

,check,passed,detail
0,five_fixed_tools_return_ok,True,5개 고정 도구 정상 응답
1,collection_and_graph_denominators_preserved,True,서울 목록·상세 수집·부분 그래프의 분모와 공식/잠정 연결 구분
2,sales_counts_preserve_missing,True,제공/미제공 분리; 0원 대체 없음
3,ratio_of_sums_matches_source,True,매출 합계와 야간 공통관측 분자·분모·행 수 일치
4,saved_model_values_unchanged,True,재적합 없이 저장한 계수·구간·업종 표본 반환
5,saved_omnibus_values_unchanged,True,05 공동 F 검정 및 Holm p값 일치
6,all_ten_sales_and_fe_tools,True,10종 공식 코드·명칭 모두 조회 가능
7,stores_500_definition_label,True,일반점포·전체점포 통제 모형 구분
8,total_stores_500_definition_label,True,일반점포·전체점포 통제 모형 구분
9,unadjusted_sensitivity_p_is_null,True,민감도 미보정 p값은 null


**완료 범위:** 고정 도구 5개, 호출 45건, 계약 검사 27개 통과. 실제 LLM 연결·자연어 파서·웹 서비스는 구현하지 않았다.